# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hazemwalid100-hub/first-assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



My rule calculates a score from 0 to 100 using traffic loss, CTR, impressions, and search position from two 45-day periods. Pages with higher scores are placed higher in the priority list.

The rule can produce these reason codes:

ACT_DECAY_REFRESH: traffic decreased significantly, so the page may need updated content.

ACT_CTR_OPTIMIZE: the page has many impressions but a low CTR, so its title or description may need improvement.

ACT_EXPANSION_TARGET: impressions increased but clicks did not increase enough, so the content may have an expansion opportunity.

ACT_MONITOR_STABLE: the page does not show a strong problem, so it should be monitored.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


I calculated a score for every page and keyword combination. Then I sorted the results from the highest score to the lowest score. The highest-scoring pages appear first because they are the pages most likely to need attention.

For each result, I saved the page URL, keyword, score, reason code, and explanation. The ranked results were saved as a CSV file so the content team can review the priority list

In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append("src")

from data_loader import DataLoader
from feature_engine import FeatureEngine
from scorer import ContentScorer
from decision_engine import DecisionEngine

loader = DataLoader("data/content_metrics.csv")
df = loader.preprocess_data(loader.load_data())

feature_engine = FeatureEngine()
signals = feature_engine.compute_signals(df)

scorer = ContentScorer()
scores = scorer.calculate_batch_scores(signals)

decision_engine = DecisionEngine()
decisions = decision_engine.make_batch_decisions(signals, scores)
ranked_results = decision_engine.prioritize_decisions(decisions)

queue = pd.DataFrame([
    {
        "rank": rank,
        "page_url": decision.page_url,
        "keyword": decision.keyword,
        "score": decision.composite_score,
        "reason_code": decision.action_flag.value,
        "rationale": decision.rationale
    }
    for rank, decision in enumerate(ranked_results, start=1)
])

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

print(f"Saved {len(queue)} ranked results to {output_path}")


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


I reviewed the top 20 pages based on their scores. For each page, I recorded its action, reason code, confidence, and what could make the recommendation wrong.

The confidence depends on the strength of the signals. A high score gives me more confidence, but the recommendation could still be wrong because of seasonality, tracking errors, search algorithm changes, or temporary traffic changes.

A page with traffic loss may not need a refresh if the loss is temporary. A low CTR may be caused by search intent, and higher impressions may only be temporary. Therefore, the content team should review the results before taking action

In [ ]:
top_20 = queue.head(20).copy()

def confidence_note(row):
    if row["score"] >= 50:
        return "High confidence because the score is strong"
    elif row["score"] >= 25:
        return "Medium confidence because the signals are noticeable"
    return "Low confidence because the score is weak"

def possible_error(row):
    if row["reason_code"] == "ACT_DECAY_REFRESH":
        return "The traffic loss may be temporary or caused by seasonality"
    elif row["reason_code"] == "ACT_CTR_OPTIMIZE":
        return "The low CTR may be caused by search intent or unusual queries"
    elif row["reason_code"] == "ACT_EXPANSION_TARGET":
        return "The impression increase may be temporary"
    return "The page may change after the analysis period"

top_20["confidence_note"] = top_20.apply(confidence_note, axis=1)
top_20["what_might_make_it_wrong"] = top_20.apply(possible_error, axis=1)

print(top_20[
    [
        "rank",
        "page_url",
        "keyword",
        "reason_code",
        "confidence_note",
        "what_might_make_it_wrong"
    ]
].to_string(index=False))


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


Some weak picks may be pages with low scores but temporary traffic changes. A page may also receive the wrong action if the data is incomplete or if the search intent changed.

I checked that the project does not use product flags or future performance data as features. The features are calculated only from the available historical clicks, impressions, CTR, and position data. The action flag is created after scoring, so it is not used as an input.

The project does not have a real future outcome label yet, so there is no future label being leaked. The leakage check passed.

In [ ]:
forbidden_fields = [
    "product_flag",
    "action_flag",
    "future_clicks",
    "future_impressions",
    "future_ctr",
    "future_position"
]

leaked_fields = [
    field for field in forbidden_fields
    if field in df.columns
]

print("Leaked fields:", leaked_fields)
assert leaked_fields == []

print("No product flags or future-window fields were used.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.